In [ ]:
# === SETUP — à exécuter une fois par session. Suivi en direct dans le terminal. ===
import os, subprocess, shutil, time
from pathlib import Path
import torch

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU  :", gpu or "aucun (CPU)")
print("torch:", torch.__version__, "| cuda dispo :", torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')

# >>> URL de ton repo (repo public, sinon token nécessaire) <<<
REPO_URL    = "https://github.com/JulesV19/cardiac-JEPA"
DRIVE_ROOT  = Path('/content/drive/MyDrive/cardiac-jepa')   # persiste entre sessions
DATA_DRIVE  = DRIVE_ROOT / 'data'                           # les 2 CSV
CACHE_DRIVE = DRIVE_ROOT / 'ptbxl_100hz.npy'                # cache signaux (~1 Go)
CACHE_LOCAL = Path('/content/ptbxl_100hz.npy')              # copie locale = lecture rapide
REPO_DIR    = Path('/content/cardiac-jepa')

# --- code : clone ou pull ---
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt
print("code prêt :", REPO_DIR)

# --- données : cache Drive -> disque local (une fois) ---
required = [CACHE_DRIVE, DATA_DRIVE / 'ptbxl_database.csv', DATA_DRIVE / 'scp_statements.csv']
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Manquant sur Drive :\n  " + "\n  ".join(missing))
if not CACHE_LOCAL.exists():
    print("copie du cache (~1 Go, une fois)...", flush=True); t0 = time.time()
    shutil.copy(CACHE_DRIVE, CACHE_LOCAL); print(f"  copié en {time.time()-t0:.0f}s")
os.environ['PTBXL_DATA_DIR'] = str(DATA_DRIVE)
os.environ['PTBXL_CACHE']    = str(CACHE_LOCAL)

from jepa.data import PTBXLDataset
ds = PTBXLDataset('pretrain'); x = ds[0]
print(f"OK — pretrain={len(ds)} ECG | échantillon {tuple(x.shape)} "
      f"(moy={x.mean():.3f} std={x.std():.3f})")

In [ ]:
# === TRAIN — ViT-JEPA (tiny.yaml) ===
# best.pt  = meilleur epoch selon la sonde linéaire (folds 1-8 -> AUROC fold 9)
# latest.pt = reprise après déconnexion (--resume auto). Aucun checkpoint périodique.
# --patience N : arrêt anticipé si la sonde ne progresse pas depuis N epochs (0 = off).
RUN_DIR = DRIVE_ROOT / 'runs' / 'tiny_v1'
RUN_DIR.mkdir(parents=True, exist_ok=True)

!python -u -m jepa.train \
    --config jepa/configs/tiny.yaml \
    --out "{RUN_DIR}" \
    --resume auto \
    --probe-subsample 4000 \
    --patience 15 \
    --workers 2